In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))  # Graph-SSL
print(sys.path[-1])
from wl_gcl.src.utils.wl_core import WLHierarchyEngine
import torch
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx
import numpy as np



C:\Users\halac\Graph-SSL


In [ ]:
from wl_gcl.src.data_loader.dataset import load_dataset
# 1. Load a molecule
dataset = load_dataset("cora", root = "../data")
data = dataset.data


In [46]:
from wl_gcl.configs.baseline import make_baseline_cfg
from dataclasses import replace
from wl_gcl.src.models import get_model

cfg = make_baseline_cfg("cora")
cfg = replace(cfg, model="gin")




In [42]:
nodes = range(data.num_nodes)
print(f"Total number of nodes : {len(nodes)}")
edges = data.edge_index.t().tolist()
engine = WLHierarchyEngine(nodes, edges)
engine.build_wl_tree(max_iterations=3)

level = 2
y_wl, cid2idx, num_classes = engine.get_level_targets(level=level)
y_wl = y_wl.to(cfg.device)
print(f"At level {level}, the number of clusters is {num_classes}")

Total number of nodes : 2708
Building WL Tree (Convergence Mode: False)...
At level 2, the number of clusters is 1589


In [43]:
def get_triplets(engine, num_samples=2000):
    triplets = []
    for _ in range(num_samples):
        anchor = torch.randint(0, data.num_nodes, (1,)).item()
        # Positive: A node that shares a cluster at T=2
        pos_pool = engine.get_cluster_at_level(anchor, level=2)
        # Negative: A node that diverged earlier (at T=1 or T=0)
        neg_pool = engine.get_hard_negatives(anchor, level=1)
        
        if pos_pool and neg_pool:
            triplets.append([anchor, np.random.choice(pos_pool), np.random.choice(neg_pool)])
    return torch.tensor(triplets).to(cfg.device)

triplet_indices = get_triplets(engine)

In [63]:
model_ce =  get_model(
        name=cfg.model,
        input_dim=dataset.num_features,
        hidden_dim=cfg.hidden_dim,
        out_dim=cfg.out_dim,
        dropout=cfg.dropout,
        tau=cfg.tau,  #gin
        num_layers=cfg.num_layers,  #gin
        heads=cfg.heads  #gat 
    )

model_triplet =  get_model(
        name=cfg.model,
        input_dim=dataset.num_features,
        hidden_dim=cfg.hidden_dim,
        out_dim=cfg.out_dim,
        dropout=cfg.dropout,
        tau=cfg.tau,  #gin
        num_layers=cfg.num_layers,  #gin
        heads=cfg.heads  #gat 
    )

In [64]:
import torch.nn.functional as F

def train_step(model, mode="triplet"):
    model.train()
    ce_classifier = torch.nn.Linear(cfg.out_dim, num_classes).to(cfg.device)
    optimizer = torch.optim.Adam(list(model.parameters()) + list(ce_classifier.parameters()), lr=0.01)
    optimizer.zero_grad()
    
    # Get GIN embeddings
    z = model(data.x.to(cfg.device), data.edge_index.to(cfg.device))
    
    if mode == "ce":
        # CLASS SEPARATION: Force nodes into discrete "color" bins
        logits = ce_classifier(z)
        loss = F.cross_entropy(logits, y_wl)
    else:
        # METRIC LEARNING: Force structural neighbors to be close
        anchor, pos, neg = z[triplet_indices[:,0]], z[triplet_indices[:,1]], z[triplet_indices[:,2]]
        loss = F.triplet_margin_loss(anchor, pos, neg, margin=1.0)
    
    loss.backward()
    optimizer.step()
    return loss.item()

# Run training for 100 epochs
for epoch in range(100):
    loss_val = train_step(model_triplet, mode="triplet") 
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss: {loss_val:.4f}")

Epoch 0 | Loss: 0.2145
Epoch 20 | Loss: 0.1052
Epoch 40 | Loss: 0.0337
Epoch 60 | Loss: 0.0244
Epoch 80 | Loss: 0.0127


In [65]:
for epoch in range(100):
    loss_val = train_step(model_ce, mode="ce") 
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss: {loss_val:.4f}")

Epoch 0 | Loss: 7.3729
Epoch 20 | Loss: 7.3717
Epoch 40 | Loss: 7.3637
Epoch 60 | Loss: 7.3706
Epoch 80 | Loss: 7.3666


In [52]:
from scipy.stats import spearmanr

@torch.no_grad()
def evaluate_geometry(model, engine, data):
    model.eval()
    z = model(data.x.to(cfg.device), data.edge_index.to(cfg.device))
    
    # Sample pairs to check correlation
    sample_idx = torch.randint(0, data.num_nodes, (500,))
    z_dist, wl_dist = [], []
    
    for i in range(len(sample_idx)):
        for j in range(i+1, len(sample_idx)):
            u, v = sample_idx[i].item(), sample_idx[j].item()
            z_dist.append(torch.norm(z[u] - z[v]).item())
            wl_dist.append(engine.get_wl_distance(u, v))
            
    # Spearman Correlation: High correlation means the embedding space 
    # honors the WL hierarchy (exactly what your paragraph argues for).
    corr, _ = spearmanr(z_dist, wl_dist)
    return corr

print(f"Structural Correlation (Metric Alignment) model ce: {evaluate_geometry(model_ce, engine, data):.4f}")
print(f"Structural Correlation (Metric Alignment) model triplet: {evaluate_geometry(model_triplet, engine, data):.4f}")

Structural Correlation (Metric Alignment) model ce: 0.2123
Structural Correlation (Metric Alignment) model triplet: 0.3022


In [62]:
from wl_gcl.src.trainers.eval import evaluate_linear_probe
evaluate_linear_probe(
        model=model_triplet,
        data=data,
        num_classes=dataset.num_classes,
        device=cfg.device
    )

0.29899999499320984